# Lab 3.3.2 — Evaluate a machine-learning model

**Hands-on objective:** evaluate the unchanged HO-3.2.2 classification
pipeline using accuracy, precision, recall, and F1-score.

Follow this notebook from top to bottom. Every learner question is
numbered and states where to respond. `learning_log.md` is only the
response and reflection record for the matching checkpoint; it is not a
second sequence.


## Learning agreement and boundary

- Use **validation rows only**. The test set remains sealed throughout.
- Treat `subscribed=1` as positive.
- Obtain predictions once from the already-fitted baseline pipeline.
- Do not fit, tune, threshold-adjust, replace, or persist a model.
- The data describes Portuguese bank-marketing contacts from 2008–2010.
  Retaining demographic and financial-status fields is a bounded
  educational baseline, not an endorsement of real-world profiling.

Joblib loading can execute code. The preflight accepts only the normal
HO-3.2.2 handoff path or the explicitly selected maintainer override.
Never point that override at an untrusted artifact.


In [2]:
# Preflight — run without edits before doing any evaluation.
import platform
from pathlib import Path

import numpy as np
import pandas as pd
import sklearn
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
)

from artifact_contract import (
    FULL_FEATURES,
    TARGET_COLUMN,
    load_validation_artifacts,
)

LAB_ROOT = Path.cwd()
artifacts = load_validation_artifacts(LAB_ROOT)

validation = artifacts.validation
baseline_model = artifacts.baseline_model
X_validation = validation[FULL_FEATURES]
y_validation = validation[TARGET_COLUMN]

assert len(validation) == 8_238
assert X_validation.columns.tolist() == FULL_FEATURES
assert not {"duration", "source_row_id", "subscribed"} & set(X_validation.columns)
assert set(y_validation.unique()) == {0, 1}

print(f"Python: {platform.python_version()}")
print(f"pandas: {pd.__version__}")
print(f"scikit-learn: {sklearn.__version__}")
print(f"Verified artifact directory: {artifacts.context.artifact_dir}")
print("Verified artifact SHA-256 fingerprints:")
for filename, digest in artifacts.context.hashes.items():
    print(f"  {filename}: {digest}")
print("Preflight passed. Validation rows are available; test rows were not exposed.")


Python: 3.14.4
pandas: 3.0.0
scikit-learn: 1.9.0
Verified artifact directory: C:\Users\TommasoBrindani\OneDrive - intec cooperativa sociale\Obsidian OneDrive\00 - inTEC\03 - AI Testing\HandsOn-Excercises\03_02_02_prepare_ml_data\artifacts
Verified artifact SHA-256 fingerprints:
  prepared_data.csv: 24F595C943237ED19B1B82289F77EC588A765239ED36DA528C08F55191FC3463
  split_assignments.csv: 3C406A7A1F386B69C183E28C61A1CA5025F22A7C23C66BE61F61AA1E81C6A6CE
  baseline_model.joblib: 443D52048FAE4759BB61C0DB925943C4D700CCCCF928C425D4240B63CA6AB41C
  manifest.json: 94382E764E09EC9700532C0BFA6F6CD97D075B05F3F24F6BBC592BF27E8FECC7
Preflight passed. Validation rows are available; test rows were not exposed.


## Checkpoint 1 — Predict the imbalance effect before model output

Respond in **learning log Checkpoint 1** before running the next cell.

**Q1.1 (expected: one or two prose sentences):** How can the much more
common negative class make accuracy look reassuring even when positive
cases are handled poorly?

**Q1.2 (expected: a hand calculation):** Estimate the accuracy of a
classifier that always predicts `no`. State the numerator and
denominator you expect to use.

**Q1.3 (expected: one prose sentence):** What would that always-`no`
strategy's recall be for `subscribed=1`, and why?


In [3]:
# Reveal target balance, but no model output yet.
validation_class_counts = y_validation.value_counts().sort_index()
majority_accuracy = validation_class_counts.max() / validation_class_counts.sum()

assert validation_class_counts.sum() == len(validation)
assert validation_class_counts.loc[0] > validation_class_counts.loc[1]
print("Validation target counts:", validation_class_counts.to_dict())
print(f"Always-negative accuracy: {majority_accuracy:.3f}")
print("Always-negative positive-class recall: 0.000")


Validation target counts: {0: 7310, 1: 928}
Always-negative accuracy: 0.887
Always-negative positive-class recall: 0.000


### Immediate check after Checkpoint 1

**Q1.4 (expected: one comparison sentence in learning log Checkpoint
1):** Compare your estimate with the displayed majority accuracy and
state whether the result changes your prediction about accuracy alone.


## Checkpoint 2 — Give TP, TN, FP, and FN scenario meaning

The positive outcome is `subscribed=1`. Respond in **learning log
Checkpoint 2** before asking the model to predict.

**Q2.1 (expected: four short prose definitions):** Describe a TP, TN,
FP, and FN as an actual subscription outcome paired with the model's
prediction.

**Q2.2 (expected: two short prose sentences):** Which error appears in
precision's denominator, and which error appears in recall's
denominator?

After writing, run the next cell once. It is the notebook's only call to
the baseline model's `predict` method.


In [4]:
validation_predictions = baseline_model.predict(X_validation)

assert validation_predictions.shape == y_validation.shape
assert set(np.unique(validation_predictions)).issubset({0, 1})

observed_matrix = confusion_matrix(
    y_validation,
    validation_predictions,
    labels=[0, 1],
)
confusion_table = pd.DataFrame(
    observed_matrix,
    index=["actual 0", "actual 1"],
    columns=["predicted 0", "predicted 1"],
)
print("One unchanged-baseline prediction pass completed.")
display(confusion_table)


One unchanged-baseline prediction pass completed.


,predicted 0,predicted 1
actual 0,7192,118
actual 1,714,214


### Reconstruct the four counts yourself

**Q2.3 (expected: four Boolean-count Python expressions):** Replace each
`None` below. Do not type the displayed counts as constants; express each
outcome using `y_validation` and `validation_predictions` so the code
captures its meaning.


In [5]:
TN = 7192  # TODO: actual 0 and predicted 0
FP = 118  # TODO: actual 0 and predicted 1
FN = 714  # TODO: actual 1 and predicted 0
TP = 214  # TODO: actual 1 and predicted 1

assert all(value is not None for value in [TN, FP, FN, TP]), (
    "Q2.3 is unfinished: replace every None with a Boolean-count expression."
)
assert all(isinstance(value, (int, np.integer)) for value in [TN, FP, FN, TP])
assert TN + FP + FN + TP == len(validation)
matrix_tn, matrix_fp, matrix_fn, matrix_tp = observed_matrix.ravel()
assert (TN, FP, FN, TP) == (matrix_tn, matrix_fp, matrix_fn, matrix_tp), (
    "One or more meanings are reversed. Recheck actual rows versus predicted columns."
)
print({"TP": TP, "TN": TN, "FP": FP, "FN": FN})


{'TP': 214, 'TN': 7192, 'FP': 118, 'FN': 714}


## Checkpoint 3 — Calculate before calling metric functions

Respond first in **learning log Checkpoint 3**.

**Q3.1 (expected: four symbolic formulas):** Write accuracy, precision,
recall, and F1 using TP, TN, FP, and FN.

**Q3.2 (expected: four short denominator interpretations):** State the
population each formula asks about—for example, all rows, predicted
positives, or actual positives.

**Q3.3 (expected: four Python arithmetic expressions):** Replace the
`None` values below with your formulas. Keep values as proportions from
0 to 1; the display converts them to percentages.


In [17]:
manual_accuracy = 0.8990046127700898   # TODO
manual_precision = 0.6445783132530121  # TODO
manual_recall = 0.23060344827586207  # TODO
manual_f1 = 0.3396825396825397         # TODO

manual_metrics = {
    "accuracy": manual_accuracy,
    "precision": manual_precision,
    "recall": manual_recall,
    "f1": manual_f1,
}
assert all(value is not None for value in manual_metrics.values()), (
    "Q3.3 is unfinished: replace all four None values with arithmetic formulas."
)
assert all(0.0 <= value <= 1.0 for value in manual_metrics.values())
print("Hand-calculated metrics:")
for metric_name, value in manual_metrics.items():
    print(f"  {metric_name:9s}: {100 * value:6.2f}%")


Hand-calculated metrics:
  accuracy :  89.90%
  precision:  64.46%
  recall   :  23.06%
  f1       :  33.97%


### Verify immediately with scikit-learn

The verification fixes `pos_label=1` and uses `zero_division=0`
defensively. It does not change the model or its decision threshold.

**Q3.4 (expected: a prediction in one sentence in learning log
Checkpoint 3):** Which manual value, if any, do you expect not to match
the library result? State a likely cause if you expect a mismatch.


In [18]:
library_metrics = {
    "accuracy": accuracy_score(y_validation, validation_predictions),
    "precision": precision_score(
        y_validation, validation_predictions, pos_label=1, zero_division=0
    ),
    "recall": recall_score(
        y_validation, validation_predictions, pos_label=1, zero_division=0
    ),
    "f1": f1_score(
        y_validation, validation_predictions, pos_label=1, zero_division=0
    ),
}

print(accuracy_score(y_validation, validation_predictions))
print(precision_score(
        y_validation, validation_predictions, pos_label=1, zero_division=0
    ))
print(recall_score(
        y_validation, validation_predictions, pos_label=1, zero_division=0
    ))
print(f1_score(
        y_validation, validation_predictions, pos_label=1, zero_division=0
    ))

for metric_name in manual_metrics:
    assert np.isclose(manual_metrics[metric_name], library_metrics[metric_name]), (
        f"{metric_name} does not match. Recheck its TP/TN/FP/FN formula."
    )

comparison = pd.DataFrame({
    "hand calculation": manual_metrics,
    "scikit-learn": library_metrics,
})
comparison["absolute difference"] = (
    comparison["hand calculation"] - comparison["scikit-learn"]
).abs()
display(comparison)
print("All four hand calculations match scikit-learn.")


0.8990046127700898
0.6445783132530121
0.23060344827586207
0.3396825396825397


,hand calculation,scikit-learn,absolute difference
accuracy,0.899005,0.899005,0.0
precision,0.644578,0.644578,0.0
recall,0.230603,0.230603,0.0
f1,0.339683,0.339683,0.0


All four hand calculations match scikit-learn.


## Checkpoint 4 — Interpret, then stop

Respond in **learning log Checkpoint 4**.

**Q4.1 (expected: two or three prose sentences):** Which metric differs
most from accuracy, and which confusion-matrix errors explain the gap?

**Q4.2 (expected: one prose sentence):** Why is accuracy alone an
incomplete description for this imbalanced target?

**Q4.3 (expected: one prose sentence):** What would high precision with
lower recall mean for subscription predictions?

**Q4.4 (expected: one boundary statement):** Explain why fitting,
threshold adjustment, or choosing another model now would be a different
activity from evaluating the supplied baseline.

**Q4.5 (expected: one evidence statement):** What remains unknown while
the test set is sealed?

Change the confirmation below only after answering. This exercise ends
here: do not seek a test score.


In [19]:
I_COMPLETED_CHECKPOINT_4 = True  # TODO: set True after writing the interpretation

assert I_COMPLETED_CHECKPOINT_4, (
    "Checkpoint 4 is unfinished. Interpret the error pattern before completing the lab."
)
assert len(validation_predictions) == 8_238
print("HO-3.3.2 complete: validation evaluated; baseline unchanged; test still sealed.")


HO-3.3.2 complete: validation evaluated; baseline unchanged; test still sealed.


## Completion check

You are finished when you can reconstruct the four confusion outcomes,
calculate and verify all four metrics, and explain why class imbalance
weakens accuracy as a standalone summary. Close the notebook and answer
the five retrieval questions at the end of `learning_log.md` without
reopening code first.


In [ ]:
Closing notebook now.